# 16장 CNN과 LeNet — 위치가 달라도 같은 모양

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `p01`~`p04` 와 `lenet.py` 를 치고 실행. `p02` 의 숫자를 종이에 먼저 계산한 뒤 실행 | [ch16_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `LeNet.forward` 각 줄 뒤에 그 줄을 지난 shape을 주석으로 | [ch16_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `LeNet` 복원. 그다음 `conv` 를 세 층으로 늘린 나만의 변형을 만들어 `p03` 의 표에 한 줄 추가 | [ch16_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "16"))                      # 이 장의 코드 폴더로 이동
if not os.path.isdir(os.path.join(os.environ["JUCOS_DATA"], "shapes")):   # 15장의 도형 데이터가 없으면 만든다
    !python ../15/make_shapes.py
print("준비 끝. 작업 폴더: code/16")

준비 끝. 작업 폴더: code/16


## `p01_conv_basics.py` — 16.1 합성곱(convolution)이 하는 일을 작은 예로 본다

In [ ]:
# 16.1 합성곱(convolution)이 하는 일을 작은 예로 본다
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

In [ ]:
# ---- 5x5 흑백 이미지 하나: 가운데 세로선
img = torch.zeros(1, 1, 5, 5)                    # [배치, 채널, 높이, 너비]
img[0, 0, :, 2] = 1.0
print(img[0, 0])

tensor([[0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0.]])


In [ ]:
# ---- 커널(필터) 3x3: "세로 경계"를 찾는 손으로 만든 필터
kernel = torch.tensor([[[[-1.0, 0.0, 1.0],
                         [-1.0, 0.0, 1.0],
                         [-1.0, 0.0, 1.0]]]])   # [출력채널 1, 입력채널 1, 3, 3]
out = F.conv2d(img, kernel)                      # 커널을 이미지 위로 밀며 곱해서 더한다
print("conv 출력 shape:", out.shape)             # [1, 1, 3, 3]: 5-3+1 = 3
print(out[0, 0])                                 # 선의 왼쪽은 +3, 오른쪽은 -3, 나머지 0

conv 출력 shape: torch.Size([1, 1, 3, 3])
tensor([[ 3.,  0., -3.],
        [ 3.,  0., -3.],
        [ 3.,  0., -3.]])


In [ ]:
# ---- padding 으로 크기 유지, stride 로 건너뛰기
print("padding=1 :", F.conv2d(img, kernel, padding=1).shape)     # [1,1,5,5]
print("stride=2  :", F.conv2d(img, kernel, stride=2).shape)      # [1,1,2,2]

padding=1 : torch.Size([1, 1, 5, 5])
stride=2  : torch.Size([1, 1, 2, 2])


In [ ]:
# ---- 풀링: 창 안의 최댓값만 남겨 크기를 반으로
x = torch.arange(16.0).reshape(1, 1, 4, 4)
print(x[0, 0])
print(F.max_pool2d(x, 2)[0, 0])                  # 2x2 창 → [2,2]

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.],
        [12., 13., 14., 15.]])
tensor([[ 5.,  7.],
        [13., 15.]])


In [ ]:
# ---- nn.Conv2d 층: 커널을 학습으로 찾는다. 파라미터 수 = 출력채널 × 입력채널 × k × k + 출력채널
conv = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
print(conv)
print("weight:", conv.weight.shape, "| bias:", conv.bias.shape,
      "| 파라미터", sum(p.numel() for p in conv.parameters()))   # 6*1*5*5 + 6 = 156
batch = torch.zeros(64, 1, 28, 28)
print("[64,1,28,28] →", conv(batch).shape)       # [64, 6, 24, 24]

Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
weight: torch.Size([6, 1, 5, 5]) | bias: torch.Size([6]) | 파라미터 156
[64,1,28,28] → torch.Size([64, 6, 24, 24])


In [ ]:
# ---- 왜 MLP 보다 나은가: 같은 28x28 입력에 대해 첫 층의 파라미터 수 비교
fc = nn.Linear(784, 6 * 24 * 24)                 # MLP 가 같은 크기의 출력을 내려면
print("MLP 첫 층 파라미터:", sum(p.numel() for p in fc.parameters()), "vs Conv2d:", 156)

MLP 첫 층 파라미터: 2712960 vs Conv2d: 156


## `lenet.py` — 16.2 LeNet 계열 CNN: 합성곱 두 번 → 펴기 → 선형 두 번. 14장 networks 패키지에 넣을 수 있는 형태.

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 16.2 LeNet 계열 CNN: 합성곱 두 번 → 펴기 → 선형 두 번. 14장 networks 패키지에 넣을 수 있는 형태.
import torch
import torch.nn as nn
import torch.nn.functional as F


class LeNet(nn.Module):
    """입력 [B, in_ch, 28, 28] (MNIST) 또는 [B, in_ch, 32, 32] (도형·CIFAR) 를 받는다."""

    def __init__(self, in_ch=1, num_classes=10, img_size=28):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 6, kernel_size=5)         # 28 → 24 (32 → 28)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)            # 12 → 8  (14 → 10)  (풀링 뒤)
        side = ((img_size - 4) // 2 - 4) // 2                     # 풀링 두 번 뒤의 한 변: 28→4, 32→5
        self.fc1 = nn.Linear(16 * side * side, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # 합성곱 → 활성화 → 풀링
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.flatten(1)                             # [B, 16*side*side]
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


class LeNetWide(LeNet):
    """구조 변경 실험 1: 채널을 두 배로. 부모의 forward 는 그대로 쓴다."""

    def __init__(self, in_ch=1, num_classes=10, img_size=28):
        super().__init__(in_ch, num_classes, img_size)
        side = ((img_size - 4) // 2 - 4) // 2
        self.conv1 = nn.Conv2d(in_ch, 12, kernel_size=5)
        self.conv2 = nn.Conv2d(12, 32, kernel_size=5)
        self.fc1 = nn.Linear(32 * side * side, 120)


class LeNetDeep(nn.Module):
    """구조 변경 실험 2: 합성곱을 한 층 더, padding 으로 크기를 유지하며 쌓기."""

    def __init__(self, in_ch=1, num_classes=10, img_size=28):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 16, 3, padding=1)          # 크기 유지
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        side = img_size // 8                                      # 풀링 세 번: 28→3, 32→4
        self.fc = nn.Linear(64 * side * side, num_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.max_pool2d(F.relu(self.conv3(x)), 2)
        return self.fc(x.flatten(1))


if __name__ == "__main__":
    for cls in (LeNet, LeNetWide, LeNetDeep):
        m = cls()
        n = sum(p.numel() for p in m.parameters())
        print(f"{cls.__name__:<10} 파라미터 {n:>8,}  [64,1,28,28] → {tuple(m(torch.zeros(64, 1, 28, 28)).shape)}")
    m = LeNet(in_ch=3, num_classes=3, img_size=32)
    print(f"LeNet(3ch,32px) 파라미터 {sum(p.numel() for p in m.parameters()):,}  [8,3,32,32] → {tuple(m(torch.zeros(8, 3, 32, 32)).shape)}")
```

In [ ]:
!python lenet.py

LeNet      파라미터   44,426  [64,1,28,28] → (64, 10)
LeNetWide  파라미터   82,518  [64,1,28,28] → (64, 10)
LeNetDeep  파라미터   29,066  [64,1,28,28] → (64, 10)
LeNet(3ch,32px) 파라미터 61,411  [8,3,32,32] → (8, 3)


## `p02_shape_trace.py` — 16.3 forward 를 한 줄씩 지나며 shape 이 어떻게 바뀌는지 추적한다 (5장 디버거로 보던 것을 코드로)

숫자를 종이에 먼저 계산한 뒤 실행하세요.

In [ ]:
# 16.3 forward 를 한 줄씩 지나며 shape 이 어떻게 바뀌는지 추적한다 (5장 디버거로 보던 것을 코드로)
import torch
import torch.nn.functional as F
from lenet import LeNet

m = LeNet()
x = torch.zeros(64, 1, 28, 28)
print("입력            ", tuple(x.shape))
x = m.conv1(x);               print("conv1 (5x5)     ", tuple(x.shape), "  28-5+1 = 24")
x = F.max_pool2d(F.relu(x), 2); print("relu+pool       ", tuple(x.shape), "  24/2 = 12")
x = m.conv2(x);               print("conv2 (5x5)     ", tuple(x.shape), "  12-5+1 = 8")
x = F.max_pool2d(F.relu(x), 2); print("relu+pool       ", tuple(x.shape), "  8/2 = 4")
x = x.flatten(1);             print("flatten(1)      ", tuple(x.shape), "  16*4*4 = 256")
x = F.relu(m.fc1(x));         print("fc1             ", tuple(x.shape))
x = F.relu(m.fc2(x));         print("fc2             ", tuple(x.shape))
x = m.fc3(x);                 print("fc3 (출력)      ", tuple(x.shape))

입력             (64, 1, 28, 28)
conv1 (5x5)      (64, 6, 24, 24)   28-5+1 = 24
relu+pool        (64, 6, 12, 12)   24/2 = 12
conv2 (5x5)      (64, 16, 8, 8)   12-5+1 = 8
relu+pool        (64, 16, 4, 4)   8/2 = 4
flatten(1)       (64, 256)   16*4*4 = 256
fc1              (64, 120)
fc2              (64, 84)
fc3 (출력)       (64, 10)


In [ ]:
# 8장 mnist_main.py 의 Net 이 fc1 입력을 9216 으로 둔 이유도 같은 계산이다:
# conv1(3x3) 28→26, conv2(3x3) 26→24, pool 24→12, 64채널 → 64*12*12 = 9216
print("8장 Net 의 fc1 입력:", 64 * 12 * 12)

8장 Net 의 fc1 입력: 9216


## `p03_compare.py` — 16.4 같은 데이터·같은 학습 루프에서 모델만 바꿔 비교: MLP vs LeNet vs LeNetWide vs LeNetDeep (MNIST, 1 에포크)

In [ ]:
# 16.4 같은 데이터·같은 학습 루프에서 모델만 바꿔 비교: MLP vs LeNet vs LeNetWide vs LeNetDeep (MNIST, 1 에포크)
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from lenet import LeNet, LeNetWide, LeNetDeep

DATA_ROOT = os.environ.get("JUCOS_DATA", "./data")
tf = transforms.ToTensor()
train_set = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=tf)
test_set = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=tf)
test_loader = DataLoader(test_set, batch_size=1000)


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1, self.fc2 = nn.Linear(784, 128), nn.Linear(128, 10)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x.flatten(1))))


def accuracy(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (model(x).argmax(1) == y).sum().item()
    model.train()
    return correct / len(test_set)


def run(cls, epochs=1, lr=0.05):
    torch.manual_seed(0)
    model = cls()
    loader = DataLoader(train_set, batch_size=64, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    t0 = time.time()
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()
    n = sum(p.numel() for p in model.parameters())
    print(f"{cls.__name__:<10} 파라미터 {n:>8,}  정확도 {accuracy(model):.4f}  ({time.time() - t0:.0f}초)")


if __name__ == "__main__":
    for cls in (MLP, LeNet, LeNetWide, LeNetDeep):
        run(cls)

MLP        파라미터  101,770  정확도 0.9615  (5초)


LeNet      파라미터   44,426  정확도 0.9798  (19초)


LeNetWide  파라미터   82,518  정확도 0.9820  (14초)


LeNetDeep  파라미터   29,066  정확도 0.9788  (17초)


## `p04_shapes.py` — 16.5 도형 데이터에서 MLP 와 LeNet: 위치가 바뀌는 입력에 합성곱이 왜 강한가

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 16.5 도형 데이터에서 MLP 와 LeNet: 위치가 바뀌는 입력에 합성곱이 왜 강한가
import os
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "15"))
from p02_custom_dataset import ImageFolderDataset
from lenet import LeNet

ROOT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "shapes")
tf = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor(), transforms.Normalize((0.5,) * 3, (0.5,) * 3)])
train_loader = DataLoader(ImageFolderDataset(os.path.join(ROOT, "train"), tf), batch_size=32, shuffle=True)
test_loader = DataLoader(ImageFolderDataset(os.path.join(ROOT, "test"), tf), batch_size=64)


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1, self.fc2 = nn.Linear(3 * 32 * 32, 128), nn.Linear(128, 3)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x.flatten(1))))


def accuracy(model):
    model.eval()
    c = 0
    with torch.no_grad():
        for x, y in test_loader:
            c += (model(x).argmax(1) == y).sum().item()
    model.train()
    return c / len(test_loader.dataset)


def run(name, model, epochs=10, lr=0.01):
    torch.manual_seed(0)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    accs = []
    for ep in range(1, epochs + 1):
        for x, y in train_loader:
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            opt.step()
        accs.append(accuracy(model))
    n = sum(p.numel() for p in model.parameters())
    print(f"{name:<8} 파라미터 {n:>8,}  에포크별 정확도 {[round(a, 2) for a in accs]}  ({time.time() - t0:.0f}초)")


if __name__ == "__main__":
    torch.manual_seed(0); run("MLP", MLP())
    torch.manual_seed(0); run("LeNet", LeNet(in_ch=3, num_classes=3, img_size=32))
```

In [ ]:
!python p04_shapes.py

MLP      파라미터  393,731  에포크별 정확도 [0.49, 0.66, 0.58, 0.66, 0.69, 0.66, 0.71, 0.71, 0.69, 0.71]  (5초)
LeNet    파라미터   61,411  에포크별 정확도 [0.75, 0.88, 0.97, 0.99, 0.93, 0.95, 0.97, 0.95, 1.0, 0.99]  (8초)


## 다음 단계

- 2단계 「주석 달기」: [ch16_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch16_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch16_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.